# Experiment 02 - Deeper CNN

**Goal:** Does increasing depth increase baseline? 
**Model:** `DeepCNN`  
**Augmentation:** None (same as baseline -> isolates the effect of depth)  
**Comparison:** Macro-F1 vs Experiment 01 on the same test split

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('.').resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import torch.nn as nn
from config import mount_drive, load_config
from augmentation import build_eval_transform
from dataset import build_dataloaders
from model import DeepCNN
from training import Trainer, compute_class_weights
from utils import set_seed, show_results, load_model, evaluate_model

mount_drive()
cfg = load_config(ROOT / 'configs/config.yaml')
set_seed(cfg.preprocessing.random_seed, cfg.device)

EXP_NAME = 'deep_cnn'
num_classes = len(cfg.classes)

eval_tf = build_eval_transform(cfg.preprocessing.image_size, use_imagenet_norm=False)

train_loader, val_loader, test_loader = build_dataloaders(
    train_transform=eval_tf,
    eval_transform=eval_tf,
    **cfg.dataloader_kwargs(),
)
print(f'Device: {cfg.device} | Dataset: {cfg.dataset_variant} | Train: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

In [ ]:
model = DeepCNN(in_channels=3, num_classes=num_classes)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DeepCNN - Trainable params: {n_params:,}')

class_weights = compute_class_weights(cfg.class_weight_source, num_classes) if cfg.training.use_class_weights else None

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=cfg.device,
    exp_name=EXP_NAME,
    results_root=cfg.results_root,
    checkpoints_root=cfg.checkpoints_root,
    num_epochs=cfg.training.num_epochs,
    learning_rate=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
    grad_clip=cfg.training.grad_clip,
    grad_accum_steps=cfg.training.grad_accum_steps,
    early_stopping_patience=cfg.training.early_stopping_patience,
    use_amp=cfg.training.use_amp,
    scheduler=cfg.training.scheduler,
    class_weights=class_weights,
    num_classes=num_classes,
)
history = trainer.train()

In [ ]:
best_ckpt = cfg.checkpoints_root / EXP_NAME / 'best.pt'
model = load_model(DeepCNN(in_channels=3, num_classes=num_classes), str(best_ckpt), cfg.device)
_, _, test_f1, test_bal, true_labels, pred_labels = evaluate_model(
    model, test_loader, cfg.device, nn.CrossEntropyLoss()
)
print(f'Test Macro-F1: {test_f1:.4f}   Balanced Acc: {test_bal:.4f}')
show_results(true_labels, pred_labels, cfg.classes, save_dir=str(cfg.results_root / EXP_NAME))